In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

In [ ]:
df = pd.read_csv('covid_toy.csv')

In [ ]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [ ]:
df['cough'].value_counts()

,count
cough,
Mild,62
Strong,38


In [ ]:
df['city'].value_counts()

,count
city,
Kolkata,32
Bangalore,30
Delhi,22
Mumbai,16


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    object 
 2   fever      90 non-null     float64
 3   cough      100 non-null    object 
 4   city       100 non-null    object 
 5   has_covid  100 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 4.8+ KB


- gender, city $\rightarrow$ nominal categorical values $\rightarrow$ apply $OneHotEncoder$

- cough $\rightarrow$ ordinal categorical values $\rightarrow$ apply $OrdinalEncoder$

- 10 Null values in $"fever"$ column $\rightarrow$ apply $SimpleImputer$

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns = ['has_covid']), df['has_covid'], test_size = 0.2)

In [ ]:
X_train

,age,gender,fever,cough,city
38,49,Female,101.0,Mild,Delhi
74,34,Female,104.0,Strong,Delhi
57,49,Female,99.0,Strong,Bangalore
34,74,Male,102.0,Mild,Mumbai
56,71,Male,NaN,Strong,Kolkata
...,...,...,...,...,...
67,65,Male,99.0,Mild,Bangalore
97,20,Female,101.0,Mild,Bangalore
99,10,Female,98.0,Strong,Kolkata
37,55,Male,100.0,Mild,Kolkata


In [ ]:
y_train

,has_covid
38,Yes
74,No
57,No
34,Yes
56,No
...,...
67,No
97,No
99,Yes
37,No


## 1. Aam Zindagi

In [ ]:
# adding simpleimputer to fever column
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.transform(X_test[['fever']])
X_train_fever.shape

(80, 1)

In [ ]:
X_test_fever

array([[ 98.],
       [101.],
       [ 98.],
       [101.],
       [100.],
       [101.],
       [100.],
       [104.],
       [101.],
       [104.],
       [100.],
       [ 98.],
       [100.],
       [ 98.],
       [ 98.],
       [102.],
       [103.],
       [104.],
       [100.],
       [103.]])

In [33]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [34]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [39]:
# One Hot Encoder -> gender, city
ohe = OneHotEncoder(drop = 'first', sparse_output = False, dtype = np.int32)
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])
X_test_gender_city = ohe.transform(X_test[['gender', 'city']])

X_train_gender_city.shape

(80, 4)

In [44]:
# Extracting Age
X_train_age = X_train[['age']].values
X_test_age = X_test[['age']].values
X_train_age.shape

(80, 1)

In [45]:
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis = 1)
X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis = 1)
X_train_transformed.shape

(80, 7)

In [52]:
X_test_transformed.shape

(20, 7)

## Mentos Zindagi

In [47]:
from sklearn.compose import ColumnTransformer

In [48]:
transformer = ColumnTransformer(transformers = [
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories = [['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output = False, drop = 'first', dtype = np.int32), ['gender', 'city'])
], remainder = 'passthrough')

In [49]:
transformer.fit_transform(X_train).shape

(80, 7)

In [50]:
transformer.transform(X_test).shape

(20, 7)